# 01 · Shot parameter extraction

Reads every shot's raw HDF5 time-series archive and reduces each diagnostic to a **95th-percentile ("peak") and mean value over the plasma window (0–6000 ms)**, ignoring non-positive samples.

**Parametrized:** Thomson-scattering core + tangential T<sub>e</sub> / n<sub>e</sub>, plasma current I<sub>p</sub>, total NBI power, total + per-gyrotron ECH power (12 gyrotrons), EFIT plasma shape (κ, upper/lower triangularity, β<sub>N</sub>, q<sub>95</sub>), and per-valve gas flow / raw signal (11 valves) → **88 columns per shot**.

**Result on the full archive:** 3,718 shots extracted, 292 failed (corrupt / truncated `.h5` files, logged to `failed_shots_v2.txt`).

*Paths are placeholders — the raw data lives on a restricted research cluster and is not included in this repository.*

In [ ]:
import h5py
import os
import numpy as np
import csv

print("Imports done")

In [ ]:
# data path, raw data stored in d3d time series path
data_dir = "/path/to/big_d3d_data/d3d_time_series_data"
all_files = os.listdir(data_dir)
all_files_sorted = sorted(all_files)

print("Total shot files found:", len(all_files_sorted))
print("First 5 files:", all_files_sorted[:5])

In [ ]:
def _get_diagnostic_series(shot_number, f, tree, key, tmin=0, tmax=6000):
    try:
        if key not in f[str(shot_number)][tree]:
            return None, None
        diag = f[str(shot_number)][tree][key]
        data = diag["data"][:]
        time = diag["dim0"][:]
        plasma_mask = (time >= tmin) & (time <= tmax)
        data_plasma = data[:, plasma_mask] if data.ndim > 1 else data[plasma_mask]
        data_nonzero = data_plasma[data_plasma > 0]
        if len(data_nonzero) == 0:
            return None, None
        return float(np.percentile(data_nonzero, 95)), float(np.mean(data_nonzero))
    except KeyError:
        return None, None

def get_electron_temperature(shot_number, f):
    return _get_diagnostic_series(shot_number, f, "D3D", r"\D3D::TOP.ELECTRONS.TS.BLESSED.CORE:TEMP")

def get_electron_density(shot_number, f):
    return _get_diagnostic_series(shot_number, f, "D3D", r"\D3D::TOP.ELECTRONS.TS.BLESSED.CORE:DENSITY")

def get_ts_tangential_density(shot_number, f):
    return _get_diagnostic_series(shot_number, f, "D3D", r"\D3D::TOP.ELECTRONS.TS.BLESSED.TANGENTIAL:DENSITY")

def get_ts_tangential_temp(shot_number, f):
    return _get_diagnostic_series(shot_number, f, "D3D", r"\D3D::TOP.ELECTRONS.TS.BLESSED.TANGENTIAL:TEMP")

def get_plasma_current(shot_number, f):
    try:
        diag = f[str(shot_number)]["PTDATA"]["IP"]
        data = diag["data"][:]
        time = diag["dim0"][:]
        mask = (time >= 0) & (time <= 6000)
        nz = data[mask][data[mask] > 0]
        if len(nz) == 0:
            return None, None
        return float(np.percentile(nz, 95)), float(np.mean(nz))
    except KeyError:
        return None, None

def get_plasma_shape(shot_number, f):
    shape_keys = {"kappa": r"\EFIT01::KAPPA", "tritop": r"\EFIT01::TRITOP",
                  "tribot": r"\EFIT01::TRIBOT", "betan": r"\EFIT01::BETAN", "q95": r"\EFIT01::Q95"}
    try:
        results = {}
        for name, key in shape_keys.items():
            if key not in f[str(shot_number)]["EFIT01"]:
                results[name] = None
                continue
            data = f[str(shot_number)]["EFIT01"][key]["data"][:]
            nz = data[data != 0]
            results[name] = float(np.mean(nz)) if len(nz) > 0 else None
        return results
    except KeyError:
        return None

NBI_BEAMS = ["NB15L", "NB15R", "NB21L", "NB21R", "NB30L", "NB30R", "NB33L", "NB33R"]

def get_nbi_power(shot_number, f):
    try:
        ref_time = None
        for beam in NBI_BEAMS:
            key = f"\\D3D::TOP.NB.{beam}:PABS"
            if key in f[str(shot_number)]["D3D"]:
                ref_time = f[str(shot_number)]["D3D"][key]["dim0"][:]
                break
        if ref_time is None:
            return None, None
        total = np.zeros(len(ref_time))
        for beam in NBI_BEAMS:
            key = f"\\D3D::TOP.NB.{beam}:PABS"
            if key not in f[str(shot_number)]["D3D"]:
                continue
            data = f[str(shot_number)]["D3D"][key]["data"][:]
            if len(data) == len(total):
                total += data
        mask = (ref_time >= 0) & (ref_time <= 6000)
        nz = total[mask][total[mask] > 0]
        if len(nz) == 0:
            return 0.0, 0.0
        return float(np.percentile(nz, 95)), float(np.mean(nz))
    except KeyError:
        return None, None

ECH_TOWERS = {"BORIS": "BOR", "CHEWBACCA": "CHE", "DOROTHY": "DOR", "HAN": "HANDL",
              "KATYA": "KAT", "LEIA": "LEI", "LION": "LIO", "LUKE": "LUK",
              "NASA": "NAS", "NATASHA": "NAT", "R2D2": "R2D", "SCARECROW": "SCA"}

def _get_ech_param_by_tower(shot_number, f, tag, label):
    results = {}
    for tower, code in ECH_TOWERS.items():
        key = f"\\D3D::TOP.RF.ECH.{tower}:EC{code}{tag}"
        peak, mean = _get_diagnostic_series(shot_number, f, "D3D", key)
        results[f"peak_{label}_{tower}"] = peak
        results[f"mean_{label}_{tower}"] = mean
    return results

def get_ech_power_by_tower(shot_number, f):
    return _get_ech_param_by_tower(shot_number, f, "FPWRC", "echpwr")

def get_ech_power(shot_number, f):
    """Summed ECH power across original 10 towers"""
    fpwr_keys = {t: f"\\D3D::TOP.RF.ECH.{t}:EC{c}FPWRC" for t, c in ECH_TOWERS.items()
                 if t not in ("HAN", "NASA")}
    try:
        total_peak, total_mean, any_fired = 0.0, 0.0, False
        for key in fpwr_keys.values():
            if key not in f[str(shot_number)]["D3D"]:
                continue
            data = f[str(shot_number)]["D3D"][key]["data"][:]
            time = f[str(shot_number)]["D3D"][key]["dim0"][:]
            mask = (time >= 0) & (time <= 6000)
            nz = data[mask][data[mask] > 0]
            if len(nz) > 0:
                total_peak += float(np.percentile(nz, 95))
                total_mean += float(np.mean(nz))
                any_fired = True
        return (total_peak, total_mean) if any_fired else (0.0, 0.0)
    except KeyError:
        return None, None

GAS_VALVES = ["GASA", "GASB", "GASC", "GASD", "GASE", "LOB1", "LOB2", "PFX1", "PFX2", "PFX3", "UOB"]

def get_gas_flow_by_valve(shot_number, f):
    results = {}
    for valve in GAS_VALVES:
        key = f"\\D3D::TOP.NEUTRALS.GASFLOW.{valve}:FLOW"
        peak, mean = _get_diagnostic_series(shot_number, f, "D3D", key)
        results[f"peak_gasflow_{valve}"] = peak
        results[f"mean_gasflow_{valve}"] = mean
    return results

def get_gas_raw_by_valve(shot_number, f):
    results = {}
    for valve in GAS_VALVES:
        key = f"\\D3D::TOP.NEUTRALS.GASFLOW.{valve}:RAW"
        peak, mean = _get_diagnostic_series(shot_number, f, "D3D", key)
        results[f"peak_gasraw_{valve}"] = peak
        results[f"mean_gasraw_{valve}"] = mean
    return results

print("Parameter functions defined")

In [ ]:
def extract_shot_parameters(shot_number, data_dir):
    filename = os.path.join(data_dir, f"{shot_number}.h5")
    if not os.path.exists(filename):
        return None
    try:
        with h5py.File(filename, "r") as f:
            if str(shot_number) not in f:
                return None
            peak_temp, mean_temp = get_electron_temperature(shot_number, f)
            peak_dens, mean_dens = get_electron_density(shot_number, f)
            peak_nbi, mean_nbi = get_nbi_power(shot_number, f)
            peak_ip, mean_ip = get_plasma_current(shot_number, f)
            peak_ech, mean_ech = get_ech_power(shot_number, f)
            shape = get_plasma_shape(shot_number, f)
            peak_tt_d, mean_tt_d = get_ts_tangential_density(shot_number, f)
            peak_tt_t, mean_tt_t = get_ts_tangential_temp(shot_number, f)
            result = {
                "shot_number": shot_number, "peak_temp": peak_temp, "mean_temp": mean_temp,
                "peak_density": peak_dens, "mean_density": mean_dens,
                "peak_nbi": peak_nbi, "mean_nbi": mean_nbi,
                "peak_ip": peak_ip, "mean_ip": mean_ip,
                "peak_ech": peak_ech, "mean_ech": mean_ech,
                "kappa": shape["kappa"] if shape else None, "tritop": shape["tritop"] if shape else None,
                "tribot": shape["tribot"] if shape else None, "betan": shape["betan"] if shape else None,
                "q95": shape["q95"] if shape else None,
                "peak_ts_tangential_density": peak_tt_d, "mean_ts_tangential_density": mean_tt_d,
                "peak_ts_tangential_temp": peak_tt_t, "mean_ts_tangential_temp": mean_tt_t,
            }
            result.update(get_ech_power_by_tower(shot_number, f))
            result.update(get_gas_flow_by_valve(shot_number, f))
            result.update(get_gas_raw_by_valve(shot_number, f))
            return result
    except Exception as e:
        print(f"Error on shot {shot_number}: {e}")
        return None
print("Master function defined")

In [ ]:
# test on two shots 
result1 = extract_shot_parameters(144187, data_dir)
result2 = extract_shot_parameters(144229, data_dir)

print("Shot 144187:")
for key, value in result1.items():
    print(f"  {key}: {value}")

print("\nShot 144229:")
for key, value in result2.items():
    print(f"  {key}: {value}")

In [ ]:
fieldnames = (
    ["shot_number", "peak_temp", "mean_temp", "peak_density", "mean_density",
     "peak_nbi", "mean_nbi", "peak_ip", "mean_ip", "peak_ech", "mean_ech",
     "kappa", "tritop", "tribot", "betan", "q95",
     "peak_ts_tangential_density", "mean_ts_tangential_density",
     "peak_ts_tangential_temp", "mean_ts_tangential_temp"]
    + [f"{s}_echpwr_{t}" for t in ECH_TOWERS for s in ("peak", "mean")]
    + [f"{s}_gasflow_{v}" for v in GAS_VALVES for s in ("peak", "mean")]
    + [f"{s}_gasraw_{v}" for v in GAS_VALVES for s in ("peak", "mean")]
)

print("Total columns:", len(fieldnames))
print(fieldnames[:10], "...")

In [ ]:
# CSV Loop 
output_file = "/path/to/outputs/shot_parameters_v2.csv"
failed_file = "/path/to/outputs/failed_shots_v2.txt"
total, success, failed = len(all_files_sorted), 0, 0

with open(output_file, "w", newline="") as csvfile, open(failed_file, "w") as failfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for i, filename in enumerate(all_files_sorted):
        if not filename.endswith(".h5"):
            continue
        name_part = filename.replace(".h5", "")
        if not name_part.isdigit():
            continue
        shot_number = int(name_part)
        if i % 200 == 0:
            print("Progress:", i, "/", total, "| succeeded:", success, "| failed:", failed)
        result = extract_shot_parameters(shot_number, data_dir)
        if result is None:
            failed += 1
            failfile.write(str(shot_number) + "\n")
            continue
        writer.writerow(result)
        success += 1

print("Succeeded:", success, "| Failed:", failed)
print("Saved to:", output_file)